# Fourteen empty cells on a T4: the two seeds left at one depth

Three fourteen-empty repairs have been running on the laptop since 2026-11-06. At
2026-11-12T21:47:46 they sat at rounds 15, 20 and 14 with `W=0` in all three, 544,080 s
(151 h) of wall clock behind them, and the log of seed 1 shows what a refit costs there:
`fit loss 0.4499 epochs 200 [83320s]` and then `fit loss 0.6754 epochs 200 [179115s]`, 23 and
50 hours of wall clock for two consecutive refits - wall clock, so part of it is this laptop
asleep, which is exactly why the number is a bracket and not a measurement.

That experiment was stopped on 2026-11-14 at rounds 16, 21 and 15 of 24, and its three
transcripts are archived outside this notebook, where they cannot drift: the folder
`runs/_localq/end14_stopped_2026-11-14/` holds each log with its sha256, the pids, and a
`summary.json` derived from them. Stopping it is what produced the evidence the arm below acts
on: all three seeds crossed the same wall at an aggregate of ~2,200-2,500 histories, after which
every refit burned its 200-epoch cap and the certificate fell from D 0.95-0.98 to D 0.19-0.26.

Nothing in this notebook changes that experiment, and nothing here can continue the laptop's
own seeds 0, 1 or 2: the engine writes its weights only when the whole loop ends, and it never
ended. What it can do is ask the same question again on a machine that can afford the refits.

What this notebook does is run two arms of the same depth, on the same published forty-root
suite and with the same budget, with the state written to disk as it goes:

1. it measures one optimizer step at the exact shape a late refit sees, on this runtime's CPU
   and on its GPU, and prints what the multiplier buys - measured, not assumed;
2. it runs the published suite and the published budget on seeds 0 and 2. Seed 0 is resumed
   from its round-22 state when its zip is uploaded, so it owes only the last two refits; seed 2
   starts from its base fit, because the laptop's run of that seed was stopped and left no
   weights. Seeds 3 and 1 are finished on this device and are not run here: seed 3 stopped at
   round 32 when the certificate returned only violations the aggregate already held, and seed 1
   spent all 24 rounds and stopped at the cap with the same property. Every arm carries one rule
   added to the refit: it keeps the epoch with the fewest mistakes instead of the last one, so a
   refit cannot hand back a worse policy than it was given, and it stops after 25 epochs without
   a new best, so a fit that is getting worse is not paid for to the cap. Section 2 states that
   rule, why the secretary rule is not the answer here, and its evidence; every JSON marks its
   arm as `protocol.published: false`;
3. it checkpoints every round and every five epochs, so a free session that ends at twelve
   hours costs at most five epochs of one refit, and re-running the notebook continues the
   same fits, optimizer state included;
4. it hands one zip per arm to the browser. No Drive, no upload of anything.

The certificate itself does not get faster here, and the notebook does not pretend otherwise:
`certify` walks every teacher tie-break below each root in pure Python and asks the policy for
one action at a time, a batch-1 forward that costs 3.8 ms of interpreter overhead and 0.115 s
of compute per 256 rows. That stays on the CPU. The GPU is used only to *find* weights. The
certificate is then computed on a CPU copy of exactly those float32 weights, so what the arm
reports is the engine's own number on the policy the run actually produced.

### Cómo se usa (para mí)

1. Runtime -> Change runtime type -> T4 GPU. Después: Runtime -> Run all.
2. La celda *Measure before spending* te dice cuánto acelera la T4 **en este runtime**, con
   números, y cuántas horas por refit se esperan. Si el multiplicador es menor que 3, no vale
   la pena y la propia celda lo dice.
3. La celda del conductor corre los dos brazos uno detrás de otro, cada uno con su tope de
   75 minutos, y va mandando un zip por brazo. Al agotarse el tope de un brazo se detiene solo,
   escribe el estado y manda el zip, y pasa al siguiente.
4. Si se cae la sesión o cierras la pestaña: sube los zips que ya tengas por la barra lateral,
   Run all otra vez. La celda del conductor encuentra el `state.pt` de cada brazo dentro de su
   zip y sigue en la misma ronda y la misma época. No se pierde nada más que lo que había
   entre dos checkpoints. La semilla 0 se reanuda desde su ronda 22 si subes su zip. Las
   semillas 3 y 1 no están en la lista: ya terminaron en la T4 (ronda 32 sin violaciones nuevas,
   y ronda 24 en el tope con todas las violaciones ya en el agregado) y reanudarlas solo
   volvería a certificar los mismos pesos.
5. Lo que produce, por brazo: `data/<brazo>.json`, los pesos, `state.pt` y, si el brazo
   termina, las dos lecturas pareadas `_verify` y `_holdoutB`.


In [ ]:
import os, pathlib, zipfile

try:
    from google.colab import files as _browser     # descarga directa en el navegador
except ImportError:
    _browser = None

CONTENT = pathlib.Path(os.environ.get('COLAB_CONTENT', '/content'))
SLUG, ARM_STEM = 't4', 'c4_end14_t4_s0'
_pushed = {}


def artefacts():
    """Everything this notebook writes inside the repo: the arm's JSON, its weights, the
    resume checkpoint and, once the arm finishes, the two paired reads the paper asks for."""
    found = [pathlib.Path('data', ARM_STEM + '.json'),
             pathlib.Path('data', ARM_STEM + '_verify.json'),
             pathlib.Path('data', ARM_STEM + '_holdoutB.json')]
    if pathlib.Path('runs', ARM_STEM).is_dir():
        found += sorted(pathlib.Path('runs', ARM_STEM).glob('*.pt'))
    return [p for p in found if p.is_file()]


def ship():
    """Zip what exists and hand it to the browser. This zip is the only way out of the runtime.

    No Drive: a mounted Drive writes into the account and keeps its own version of the run,
    and the run's provenance is the file, its hash and the commit in the bundle, not a path
    under My Drive. The signature uses size and mtime because `state.pt` is rewritten at every
    checkpoint: the browser is only asked again when something actually changed.
    """
    found = artefacts()
    sig = {(p.as_posix(), p.stat().st_size, int(p.stat().st_mtime)) for p in found}
    if not found or sig == _pushed.get(SLUG):
        print('nada nuevo que enviar')
        return None
    CONTENT.mkdir(parents=True, exist_ok=True)
    archive = CONTENT / ('c4_%s_results.zip' % SLUG)
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in found:
            z.write(p, arcname=p.as_posix())
    _pushed[SLUG] = sig
    size = archive.stat().st_size / 2 ** 20
    if _browser is None:
        print('%d artefactos -> %s (%.2f MB)' % (len(found), archive, size))
    else:
        print('%d artefactos -> el navegador descarga %s (%.2f MB)'
              % (len(found), archive.name, size), flush=True)
        try:
            _browser.download(str(archive))
        except Exception as exc:
            print('  AVISO: no se pudo enviar al navegador (%s); el zip quedó en %s'
                  % (exc, archive))
    return archive


print('salida: c4_%s_results.zip al navegador. Sin Drive, sin subir nada.' % SLUG)


## 0. The code this notebook runs

The cell below is the bundle: the four modules the arm imports, the fourteen-empty suite and its blind twin, as a base64 zip with the sha256 of every file and of the archive. The setup cell under it rebuilds a working copy and verifies both hashes, so a runtime needs no GitHub sign-in, no token and no upload - and a bundle that was edited by hand is refused rather than run.


In [ ]:
# The bundle: this notebook carries the code it runs, so a runtime needs nothing
# from GitHub - no authorisation, no token, no upload. Built by
# colab/build_bundle.py from commit 0f01d72. Every file's sha256 is here and
# the setup cell re-computes them, so a hand-edited or stale bundle falls back to
# the clone instead of running modified code.
import base64 as _b64, hashlib as _hl, io, zipfile

BUNDLE_COMMIT = '0f01d72'
BUNDLE_SHA256 = '69405b3454b43f12f59fe9508fbe9543ce4de85c876cdec273f227220247423d'
BUNDLE_FILES = {
    "src/c4_game.py": "f9d0dece4145b957b461e59427faea7ebcdafedf4f28823facac1236f7bd55c4",
    "src/c4_repair.py": "5e5de985ecb0c059ccfd5d893db06d47e303222cbb30d3158efea22174c7892c",
    "src/c4_endgame.py": "d40a600ca87e1d38de42ea6ab3afea8d78cef3b1f0bedee31d9d118070ec9204",
    "src/c4_endgame_verify.py": "47009f16a1649db948b0f8fcafce4d688864c8cc10c1762fd6d89d549a92ade7",
    "data/c4_end10_s1.json": "4189ccc5599c3495a11899368b7b0647699080654d3801ae4cb5359f15d7ef63",
    "data/c4_endgame_roots_10_40_holdoutB.json": "274c14a6a2d402defa9fba26373797cf44d9d97cec0ef490a825de3a062e4a4a",
    "data/c4_endgame_roots_10_40_s0.json": "3079b51610d93dd269e5104c539b63782f34f3a980199867cd6e5bab75055a00",
    "data/c4_endgame_roots_12_40_holdoutB.json": "e671bf8c7cd7b6df32643a9653c529c1cade2588226eb36597eb5ace6b786e5a",
    "data/c4_endgame_roots_12_40_s0.json": "3ef57d6feccd9a492fa99c3258b38458ace19205b7d9aebc215d39548f0d7c10",
    "data/c4_endgame_roots_14_40_holdoutB.json": "3530ecc16d6dc659763f0cc21b6ff50199d9ae404a685cd6692594e35db99e61",
    "data/c4_endgame_roots_14_40_s0.json": "006d1ced87af527ecdbb7bd8e8e8e9a2980e0316b809c1a4d20cfc8b3cc4a18a",
    "data/c4_endgame_roots_8_40_holdoutB.json": "a829bf5c47e30a0ebe671c9d4e3c1c5103dbb90b97d41e0f0cd396165dcc1082",
    "data/c4_endgame_roots_8_40_s0.json": "d3c67bde3ce07ca68384b453d58574798f9425bd2e5f2aa2118be07ac65fccc4",
    "data/c4_endgame_roots_10_80_AplusB.json": "b9ce69507eb481da8d82ed5cbb0dc88b16ba58ea22c7a36d4daeb2741120d879",
    "data/c4_endgame_roots_8_80_AplusB.json": "68e02bedc82fe8837175fdc854ab2fca9f438ee33c6850febe67bfeb4c296a24"
}
BUNDLE_B64 = (
    '...'
    'UEsFBgAAAAAPAA8AiQQAADhxBQAAAA=='
)

_archive = _b64.b64decode(BUNDLE_B64)
assert _hl.sha256(_archive).hexdigest() == BUNDLE_SHA256, 'the bundle is corrupt'
_zf = zipfile.ZipFile(io.BytesIO(_archive))
assert sorted(_zf.namelist()) == sorted(BUNDLE_FILES), 'the bundle holds other files than it claims'
for _n in _zf.namelist():
    assert _hl.sha256(_zf.read(_n)).hexdigest() == BUNDLE_FILES[_n], 'bundle was modified: ' + _n
print('this notebook carries its own code: commit %s, %d files, %.0f KB unpacked,'
      ' %.0f KB inside the notebook - no GitHub sign-in needed'
      % (BUNDLE_COMMIT, len(BUNDLE_FILES),
        sum(z.file_size for z in _zf.infolist()) / 1024, len(_archive) / 1024))


In [ ]:
import base64, hashlib, io, os, pathlib, re, shutil, subprocess, sys, zipfile

URL = 'https://github.com/starlyn2010/fragility-paper.git'
REPO_SLUG = 'starlyn2010/fragility-paper'
MARKER = pathlib.Path('src') / 'c4_endgame.py'
CONTENT = pathlib.Path(os.environ.get('COLAB_CONTENT', '/content'))
TOKEN = ''  # optional: a read-only GitHub token, only if you prefer it over the bundle


def _find_repo():
    """Any directory under /content that already holds the repo, wherever it was put."""
    seen = [pathlib.Path.cwd(), CONTENT / 'fragility-paper']
    if CONTENT.is_dir():
        seen += sorted(CONTENT.glob('*/'))
    return next((c for c in seen if (c / MARKER).exists()), None)


def _unzip_repo():
    """A repo archive in Drive or uploaded to the runtime counts as a copy of the repo.

    Two layouts have to work: an archive made by GitHub, which wraps everything in one
    folder, and `git archive`, which stores the paths flat from the repo root. So the
    candidate is the extraction directory itself as well as each top-level entry.
    """
    if not CONTENT.is_dir():
        return None
    for z in sorted(CONTENT.glob("**/*fragility-paper*.zip")):
        with zipfile.ZipFile(z) as fh:
            names = fh.namelist()
            fh.extractall(z.parent)
        tops = sorted({n.split("/")[0] for n in names if "/" in n})
        roots = [z.parent] + [z.parent / t for t in tops]
        hit = next((r for r in roots if (r / MARKER).exists()), None)
        if hit:
            print("unzipped repo archive ->", hit)
            return hit
        print("unzipped", z.name, "but it has no", MARKER)
    return None


def _unbundle():
    """Rebuild a working copy of the repository from the cell that carries it.

    The bundle cell above holds the four modules this arm runs, the fourteen-empty suite and
    its blind twin, so a runtime with the notebook needs nothing else: no GitHub
    authorisation, no token, no upload. Nothing is trusted on the way in - the archive's own
    hash and every file's hash recorded in the bundle are recomputed here, and any mismatch
    returns None so bootstrap falls through to the clone instead of running code that was
    edited inside the notebook.
    """
    if "BUNDLE_B64" not in globals():
        return None
    archive = base64.b64decode(BUNDLE_B64)
    if hashlib.sha256(archive).hexdigest() != BUNDLE_SHA256:
        print("the notebook's bundle does not match its own hash; ignoring it")
        return None
    target = CONTENT / "fragility-paper"
    with zipfile.ZipFile(io.BytesIO(archive)) as fh:
        names = sorted(fh.namelist())
        if names != sorted(BUNDLE_FILES):
            print("the notebook's bundle holds other files than it claims; ignoring it")
            return None
        for name in names:
            blob = fh.read(name)
            if hashlib.sha256(blob).hexdigest() != BUNDLE_FILES[name]:
                print("the notebook's bundle was modified:", name)
                return None
            out = target / name
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(blob)
    print("built a working copy of the repo from the notebook itself ->", target)
    return target if (target / MARKER).exists() else None


def _gh_ready():
    """True if the runtime's gh CLI is installed and already holds credentials."""
    if shutil.which('gh') is None:
        return False
    return subprocess.run(['gh', 'auth', 'status'], capture_output=True,
                          text=True).returncode == 0


def _gh_device_login():
    """Ask GitHub to authorise THIS runtime, with a code you type on any device.

    No browser exists in a runtime, so BROWSER is a command that does nothing, and gh paints
    progress with ANSI cursor moves that the notebook's output frame chokes on, so the stream
    is cleaned line by line as it arrives.
    """
    bs = chr(92)
    ansi = re.compile(chr(27) + bs + "[" + "[0-9;?<>!]*" + "[A-Za-z~]"
                      + "|" + chr(27) + "[()]" + "[A-Za-z]"
                      + "|" + chr(27) + "[=<>]")
    print('logging this runtime in to GitHub (device flow).', flush=True)
    p = subprocess.Popen(['gh', 'auth', 'login', '--hostname', 'github.com',
                          '--git-protocol', 'https', '--web'],
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'BROWSER': 'echo'})
    try:
        p.stdin.write(chr(10))
        p.stdin.flush()
    except Exception:
        pass
    for raw in iter(p.stdout.readline, ''):
        line = "".join(c for c in ansi.sub("", raw) if c == chr(10) or ord(c) >= 32)
        if line.strip():
            print(line.strip(), flush=True)
    p.wait()
    return _gh_ready()


def _clone(url):
    target = CONTENT / 'fragility-paper'
    if target.exists():
        shutil.rmtree(target)
    if shutil.which('gh') and _gh_ready():
        print("cloning with the runtime's stored GitHub credentials")
        r = subprocess.run(['gh', 'repo', 'clone', REPO_SLUG, str(target),
                            '--', '--depth', '1'], capture_output=True, text=True)
        if not r.returncode and (target / MARKER).exists():
            return target
        sys.stderr.write((r.stderr or r.stdout).strip()[-400:] + '\n')
        return None
    if TOKEN and '://' in url:
        head, rest = url.split('://', 1)
        url = head + '://x-access-token:' + TOKEN + '@' + rest
    print('fetching the repo ->', url)
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(target)],
                       capture_output=True, text=True, env=env)
    if r.returncode:
        sys.stderr.write((r.stderr or r.stdout).strip()[-400:] + '\n')
        return None
    return target if (target / MARKER).exists() else None


def bootstrap(url=URL, ask_gh=True):
    """Find the repo, get it if this runtime has none, and make `src` importable.

    A Colab kernel leaves /content on sys.path and %cd does not change it, so the repo root
    has to be inserted by hand, and so does its src directory: the modules under src import
    each other by their flat names (import c4_game), so a cell that reaches for src.c4_game
    loads a SECOND copy of the module and then configures the copy the sampler is not using.
    """
    repo = _find_repo() or _unbundle() or _unzip_repo() or _clone(url)
    if repo is None and ask_gh and not _gh_ready():
        if shutil.which('gh'):
            _gh_device_login()
            repo = _clone(url)
    if repo is None:
        raise SystemExit(
            'no copy of the repo could be made, and the bundle that normally removes that '
            'need was not usable. Run the cell titled "The bundle" just above this one, then '
            'this one; it rebuilds the working copy from the notebook itself. If that cell is '
            'missing, this notebook is an old copy - re-upload colab/c4_end14_t4.ipynb.')
    os.chdir(repo)
    for entry in (str(repo), str((repo / 'src').resolve())):
        if entry not in sys.path:
            sys.path.insert(0, entry)
    return repo.resolve()


REPO = bootstrap()

import torch as _torch
print('repo at', REPO, '| on sys.path:', str(REPO) in sys.path)
print('torch', _torch.__version__, '| compiled cuda', _torch.version.cuda,
      '| available:', _torch.cuda.is_available(),
      '|', _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else 'no GPU')


## 1. Measure before spending

A refit at this depth is not a training run anybody would call small: round 15 of seed 2 holds
2,781 aggregate histories, `agg_repeat=60` turns them into 166,860 training items, `replay`
adds 4,411, and `_batched` cuts the 171,271 rows into 669 batches of 256. Two hundred epochs
is 133,800 optimizer steps. On this laptop, one thread and two other arms competing for the
same two cores, a step at that shape measured 1.358 s, which is the order of the 23 and 50
hours the log above shows; forward alone was 0.115 s of that, so the backward pass is eleven
times the cost of the forward and this is compute, not Python. What a T4 changes is the cost
of those steps and nothing else.

That is the one part of this experiment a GPU can shorten. The next cell measures the same
step here, at full sequence length (42 tokens: the 28-move prefix, fourteen empties and the
start token), on the runtime's CPU with one thread and on its GPU, and prints the multiplier
and the projected hours. It is a measurement of this runtime, not a claim about T4s in
general: what matters is the number printed below, because that is what decides whether the
run is worth starting.


In [ ]:
import time

import torch

import c4_game as g
from c4_repair import GRUPolicy

torch.set_num_threads(1)   # one thread, as the laptop arms run
g.configure(6, 7, 4)       # the geometry the arm runs, before any model is built from it


def bench(dev, steps=6, warm=2, batch=256, seq=42):
    """One optimizer step at the shape a late refit actually sees.

    256 is `_batched`'s batch, 42 is the longest history in the fourteen-empty suite, and every
    row is fed at full length, so this is the heavy end of an epoch rather than its average.
    Same model, same Adam, same masked cross-entropy as `fit`.
    """
    pol = GRUPolicy(128, 16).to(dev)
    opt = torch.optim.Adam(pol.parameters(), lr=2e-3)
    toks = torch.randint(0, g.COLS + 2, (batch, seq), device=dev)
    lens = torch.full((batch,), seq, dtype=torch.long)
    tgt = torch.full((batch, g.COLS), 1.0 / g.COLS, device=dev)
    msk = torch.ones((batch, g.COLS), device=dev)

    def one():
        lg = pol(toks, lens).masked_fill(msk == 0, -1e9)
        loss = -(tgt * torch.log_softmax(lg, dim=-1)).sum(dim=-1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()

    for _ in range(warm):
        one()
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t = time.time()
    for _ in range(steps):
        one()
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - t) / steps


print('a late refit is 669 batches of 256 a epoch; 200 epochs is 133,800 steps')
cpu_s = bench(torch.device('cpu'))
print('cpu, 1 thread : %.3f s/step' % cpu_s)
if not torch.cuda.is_available():
    print('no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU, then re-run')
else:
    gpu_s = bench(torch.device('cuda'))
    print('gpu           : %.3f s/step  (%.1fx the CPU above)' % (gpu_s, cpu_s / gpu_s))
    if cpu_s / gpu_s < 3:
        print('  the multiplier is small: at this ratio the T4 does not pay for the move,')
        print('  and the honest thing is to say so rather than spend a session on it')
    for name, items in (('base fit (4,411 items)', 4411),
                        ('refit at round 15 (171,271)', 4411 + 2781 * 60)):
        b = -(-items // 256)
        print('  %-29s %3d batches/epoch -> %5.1f h per 200-epoch refit on GPU'
              ' (%5.1f h on CPU)' % (name, b, b * 200 * gpu_s / 3600, b * 200 * cpu_s / 3600))
del bench


## 2. The arms, and what the certificate is a statement about

The arms this notebook runs are `c4_end14_t4_s0` and `c4_end14_t4_s2`: fourteen empty cells,
six rows by seven columns, four in a row, the published suite
`data/c4_endgame_roots_14_40_s0.json` - the
same forty roots every arm in the paper at this depth reads. The suite's sha256 is checked
against the one the bundle records before a single step runs, because two arms are only
comparable if they were certified over the same roots, and a runtime that sampled its own
would be answering a different question.

Seeds 0 and 2 carry the numbers the laptop's stopped runs had, and this notebook runs them
rather than continuing them: the engine writes its weights only when the loop ends, and those
loops never did. Seed 0's first T4 session already reached round 22 in its 75 minutes, so its
zip resumes the fit that was in flight; seed 2 starts from its base fit. The two seeds that are
finished are not run here, and their numbers are what the last two seeds are measured against:
seed 3 took two sessions to round 32, where `stop = "no new violations"` with 91 violations and
every one of them already in the 3,030-history aggregate, so there was no move left to spend; it
ended at `D 0.7678` with 15 of 40 roots drawn. Seed 1 spent all 24 rounds in a single session of
61.2 minutes and stopped at the cap at `D 0.7845` with 93 violations, again all of them already
in its 2,513-history aggregate. Four seeds at one depth are what turns "the plateau belongs to
the depth and not to the laptop's clock" from an inference off three unfinished logs into a
measurement; and if one of the two remaining seeds certifies, the paper gains a finished
fourteen-empty certificate no laptop could produce this month. Both outcomes are worth the
session, which is the only reason the session is worth spending.

The budget is the one `colab/reads_plan.py` hands the laptop arms (`arm_budget(14)`):
`epochs=200, lr=2e-3, agg_repeat=60, stop=both, max_iters=24, branch=3, hidden=128, emb=16`.
It is repeated in the cell because the bundle carries `src/` and `data/` and not `colab/`; an
arm that ran a different budget would not be the same test.

### The one rule this arm adds to the published protocol

The published refit is all-or-nothing: it stops early only when every stop item is exactly
optimal (`--stop both`), and when it does not get there it keeps whatever weights the last
epoch left. The two are the same thing while the refit converges, and they are not the same
thing when it stops converging, which is what the laptop's three logs show. Seed 0 sat at
`D 0.9631` with 61 violating histories; it added those 61, the refit ran its whole 200-epoch
cap and ended at `fit loss 0.4357`, and the arm was left at `D 0.2487`. Seed 1 went from
`D 0.9742` to `0.1886` the same way, seed 2 from `0.9225` to `0.1073`, and none of the three
recovered, because the round that follows starts from the worse policy. That is days of wall
clock spent making an arm worse.

So these arms' refits keep the **best** epoch instead of the last: with the same items, the
same masks and the same test the arm already uses, it scores each epoch by its mistakes on the
stop set, keeps the weights that minimise it, and returns those; if no epoch beats the weights
the refit started from, it returns those unchanged. The only thing that changed is that a refit
can no longer hand back a worse policy than it was given - and it is a change: these arms are
not a rerun of the published protocol, and their JSON says `protocol.published: false` next to
the rule in words.

What the rule cannot do is make a certificate. It does not touch how violations are chosen or
added, and it cannot turn a plateau into a proof. If the aggregate at fourteen empty cells is
beyond what the fit can satisfy, an arm will sit at its best `D` until `max_iters` and report
that; a plateau is the honest answer to the question the laptop arms raised, and it is a
cleaner one than a collapse.

### Why the fit stops early instead of running to the cap

Keeping the best epoch is half of the rule; the other half is not paying for 200 epochs when
the best one was epoch 3. The laptop is explicit about the price: the three capped refits of
seed 1 took 83,320 s, 179,115 s and 214,729 s - 23, 50 and 60 hours each - and every one of
them ended worse than it started.

The obvious name for "when to stop looking" is the secretary rule: reject the first 37% of the
candidates and take the first one that beats the best of those. It does not apply here, for
three reasons that are properties of training rather than of taste. The rule assumes the
candidates arrive in a random order and that the choice is irreversible; epochs are a learning
curve, ordered by construction, and the best weights can be kept for the price of one copy on
the GPU, which this fit already pays. It assumes a candidate's value is known only through its
rank among those seen, while every epoch's mistake count is measured outright. And its
guarantee is about picking the single best of a fixed list, while the requirement here is only
"never worse than the weights handed in", which keeping the best satisfies with certainty. The
arithmetic is also against it: 37% of 200 epochs is 74, longer than the longest fit any
converged round needed (2-24 epochs, and 28 for a base), and the rule would then stop at the
first improvement - more epochs than the cap it replaces, with worse weights.

What does carry over is the part that reads the trend. The fit stops after `patience` epochs
without a new best, and never before `min_epochs`, so a fit that improves at least once every
25 epochs is never cut: on the laptop's crossing the best epoch was the one it started from,
and the same rule would have spent 25 epochs instead of 200. Both numbers are in `PROTOCOL`,
both travel in the JSON, and every fit records its per-epoch mistake trace, so a reader can
see the shape that justified them - and a run in which patience cut a fit that was still
improving is visible in that trace rather than silent. `keep_best=False` disables the keep-best
rule and the early stop together and restores the published refit, cap included.

The split between GPU and CPU is the point to be careful about:

* the **refits** run on the GPU - that is the 23-to-50-hours-per-round part;
* the **certificate** runs on the CPU, on a `GRUPolicy` loaded with a CPU copy of exactly the
  weights the GPU produced. `certify` is the engine's own function, imported from the bundled
  `src/c4_endgame.py`, and `act` is batch-1 by design: at 3.8 ms per call it is interpreter
  overhead, and a GPU launch would make it slower, not faster;
* the **fit's stopping test and its choice of epoch** are the only things reimplemented, in the
  driver below: the test is the same `--stop both` predicate, batched, and the choice is the
  best-epoch rule above. Neither decides what the arm reports - the reported `W`, `B`, `D`, the
  drawn count and the violations all come from `certify`.

So the arm is defined by its weights, not by the device that found them. This also means it is
not a bit-for-bit rerun of any laptop arm: a different device is a different float path, and
the JSON says so in `environment.device` and `environment.deterministic`.


In [ ]:
# ---- the arm: fourteen empty cells on this runtime's GPU --------------------------------
# Everything below is the engine's protocol (src/c4_endgame.py and src/c4_repair.py, the files
# the bundle above carries) with three additions that live only in this notebook: the refits run
# on the GPU, the state is written to disk as it goes so a session that ends costs at most the
# epochs since the last checkpoint, and a refit hands back its best epoch instead of its last
# one. PROTOCOL below is the third, in the words the JSON repeats. No file of the repository is
# edited.

import hashlib, json, os, pathlib, platform, random, time, zipfile
from datetime import datetime, timezone

import torch

import c4_game as g
from c4_repair import GRUPolicy, _batched, items_from, n_optimal_mistakes
from c4_endgame import certify, root_items

ARM, E, SEED = 'c4_end14_t4_s0', 14, 0
SUITE = 'data/c4_endgame_roots_14_40_s0.json'
SUITE_B = 'data/c4_endgame_roots_14_40_holdoutB.json'
# colab/reads_plan.py:arm_budget(14), the numbers the laptop arms run. The bundle carries
# src/ and data/ and not colab/, so this is a copy - and a copy that drifts would be a
# different test, which is why the cell prints it next to the suite it applies to.
BUDGET = {'branch': 3, 'hidden': 128, 'emb': 16, 'epochs': 200, 'lr': 2e-3,
          'agg_repeat': 60, 'stop': 'both', 'max_iters': 24}
# The departures from the published arms, stated once so the JSON can quote them verbatim.
# Evidence in section 2: seeds 0-2 each lost a policy at D 0.96-0.98 when a refit that could not
# reach exact optimality kept its last epoch anyway, and each of those capped refits cost 23-60
# hours on the laptop, all of it spent making the fit worse.
PROTOCOL = {'keep_best': True,
            'patience': 25,
            'min_epochs': 25,
            'published': False,
            'rule': ('each refit keeps the epoch with the fewest mistakes on the stop set the '
                     'arm already uses; if no epoch beats the weights it started from, it '
                     'returns those, so a refit never hands back a worse policy than it was '
                     'given. A fit that goes 25 epochs without a new best stops there, never '
                     'before epoch 25, instead of spending the rest of its 200-epoch cap on a '
                     'fit that is getting worse')}

ROOT = pathlib.Path(REPO) if 'REPO' in globals() else pathlib.Path('.').resolve()
STATE_PATH = ROOT / 'runs' / ARM / 'state.pt'
OUT_JSON = ROOT / 'data' / (ARM + '.json')
CKPT = ROOT / 'runs' / ARM / 'c4_gru.pt'
BASE_CKPT = ROOT / 'runs' / ARM / 'c4_gru_base.pt'

torch.set_num_threads(1)      # same arithmetic as the laptop arms, so the same tie-breaks
g.configure(6, 7, 4)

if torch.cuda.is_available():
    DEV = torch.device('cuda')
    _p = torch.cuda.get_device_properties(0)
    DEVICE = '%s sm_%d%d %.1f GB' % (torch.cuda.get_device_name(0), _p.major, _p.minor,
                                     _p.total_memory / 2 ** 30)
else:
    DEV, DEVICE = torch.device('cpu'), 'cpu'
print('device', DEVICE, '| torch', torch.__version__, '| cuda', torch.version.cuda)


def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def cpu_state(policy):
    return {k: v.detach().to('cpu').clone() for k, v in policy.state_dict().items()}


def cpu_opt(opt):
    sd = opt.state_dict()
    return {'state': {i: {k: (v.detach().to('cpu').clone() if torch.is_tensor(v) else v)
                          for k, v in s.items()} for i, s in sd['state'].items()},
            'param_groups': sd['param_groups']}


ST = {}          # the arm's persistent state, rewritten atomically at every checkpoint


def save(**kw):
    ST.update(kw)
    ST['when_utc'] = now()
    STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = STATE_PATH.with_suffix('.tmp')
    torch.save(ST, tmp)
    os.replace(tmp, STATE_PATH)     # a runtime that dies mid-write leaves the old file intact


def twin_of(policy):
    """The CPU copy the certificate is computed on: identical float32 weights, engine code."""
    t = GRUPolicy(BUDGET['hidden'], BUDGET['emb'])
    t.load_state_dict(cpu_state(policy))
    t.eval()
    return t


def stop_mistakes(policy, items, batch=256):
    """`n_optimal_mistakes` from src/c4_repair.py, batched on DEV: the fit's own test.

    Same items, same masks, one argmax per row, and the GRU is fed through
    pack_padded_sequence exactly as `fit` feeds it, so padding cannot reach the hidden state.
    It decides when a refit may stop and which epoch it keeps; every number the arm reports
    comes from `certify`, which is the engine's own code on the CPU twin.
    """
    was = policy.training
    policy.eval()
    bad = 0
    with torch.inference_mode():
        for idx, toks, lens in _batched(items, batch):
            masks = torch.stack([items[j]['mask'] for j in idx]).to(DEV)
            logits = policy(toks.to(DEV), lens).masked_fill(masks == 0, -1e9)
            pick = logits.argmax(dim=-1).tolist()
            for row, j in enumerate(idx):
                if pick[row] not in items[j]['optimal']:
                    bad += 1
    policy.train(was)
    return bad


class Paused(Exception):
    """The clock ended this session, not the training."""


def fit_res(policy, items, epochs, lr, seed, stop_items, tag, deadline, save_every=5,
            keep_best=None):
    """src/c4_repair.py's `fit`, on DEV, checkpointed, keeping the best epoch.

    Identical in everything else that decides the result: Adam at the same learning rate, one
    shuffle per epoch from `random.Random(seed)`, the same 256-row batches from `_batched`, the
    same masked cross-entropy, the same per-epoch stopping test. The differences are the device
    the tensors live on, that the model, the optimizer's moments and the shuffle stream are
    written to disk every `save_every` epochs (a resume continues this fit, not a new one), and
    that the weights returned are the epoch with the fewest mistakes on `stop_items` - the
    weights the fit started from included - so a refit cannot return a worse policy than it was
    given. A fit that goes `patience` epochs without a new best stops there instead of spending
    the rest of the cap; PROTOCOL above says why, `keep_best=False` reproduces the published
    refit, and section 2 carries the evidence.

    Returns (last_epoch_loss, epochs_run, info). info['entry'], ['best'] and ['final'] count
    mistakes on the stop set; info['kept'] names what came back: 'optimal' when a zero was
    reached, 'last' when the last epoch was the best, 'best@<epoch>' otherwise, or 'incumbent'
    when even the first epoch was worse than the weights handed in. info['stop'] says why the
    loop ended ('optimal', 'patience' or 'cap') and info['trace'] is the per-epoch mistake
    count, so the shape of a fit can be read out of the JSON afterwards.
    """
    if keep_best is None:
        keep_best = PROTOCOL['keep_best']
    start, opt_state, rng_state, last = 0, None, None, 0.0
    resuming = ST.get('phase') == 'fit' and ST.get('tag') == tag
    same_fit = keep_best and ST.get('best_tag') == tag and 'best' in ST
    trace = []
    if resuming:
        policy.load_state_dict(ST['model'])
        start, opt_state = ST['epoch'], ST['opt']
        rng_state, last = ST['rng'], ST['last']
        trace = list(ST.get('trace') or [])
        print('    continuing the fit of %s at epoch %d/%d' % (tag, start, epochs), flush=True)
    best_m = best_ep = best_sd = entry_m = None
    if same_fit:
        best_m, best_ep, best_sd = ST['best_m'], ST['best_ep'], ST['best']
        entry_m = ST.get('entry_m')
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    if opt_state is not None:
        opt.load_state_dict(opt_state)
    rng = random.Random(seed)
    if rng_state is not None:
        rng.setstate(rng_state)
    if keep_best and stop_items is not None and best_sd is None:
        entry_m = best_m = stop_mistakes(policy, stop_items)
        best_ep, best_sd = start, cpu_state(policy)
    ran, stop_by = start, 'cap'
    policy.train()
    for ep in range(start, epochs):
        shuffled = items[:]
        rng.shuffle(shuffled)
        total, nb = 0.0, 0
        for idx, toks, lens in _batched(shuffled, 256):
            logits = policy(toks.to(DEV), lens)
            targets = torch.stack([shuffled[j]['target'] for j in idx]).to(DEV)
            masks = torch.stack([shuffled[j]['mask'] for j in idx]).to(DEV)
            logits = logits.masked_fill(masks == 0, -1e9)
            loss = -(targets * torch.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss.detach())
            nb += 1
        last = total / max(1, nb)
        if stop_items is not None:
            m = stop_mistakes(policy, stop_items)
            trace.append(m)
            if entry_m is None:
                entry_m = m
            if keep_best and (best_m is None or m < best_m):
                best_m, best_ep, best_sd = m, ep + 1, cpu_state(policy)
            if m == 0:
                print('    every stop item optimal after epoch %d/%d' % (ep + 1, epochs),
                      flush=True)
                save(phase='fit', tag=tag, epoch=ep + 1, model=cpu_state(policy),
                     opt=cpu_opt(opt), rng=rng.getstate(), last=last, best_tag=tag,
                     best_m=best_m, best_ep=best_ep, best=best_sd, entry_m=entry_m,
                     trace=trace)
                return last, ep + 1, {'entry': entry_m, 'best': best_m, 'final': m,
                                      'kept': 'optimal', 'stop': 'optimal', 'trace': trace}
        ran = ep + 1
        over = bool(deadline) and time.time() > deadline
        if (ep + 1) % save_every == 0 or ep + 1 == epochs or over:
            save(phase='fit', tag=tag, epoch=ep + 1, model=cpu_state(policy),
                 opt=cpu_opt(opt), rng=rng.getstate(), last=last, best_tag=tag,
                 best_m=best_m, best_ep=best_ep, best=best_sd, entry_m=entry_m, trace=trace)
        if ep % 10 == 0 or ep + 1 == epochs:
            print('    epoch %3d/%d loss %.4f%s'
                  % (ep + 1, epochs, last,
                     '' if stop_items is None else ' mistakes %d' % m), flush=True)
        if over:
            raise Paused('the budget ended after epoch %d/%d of fit %s'
                         % (ep + 1, epochs, tag))
        if (keep_best and stop_items is not None and best_ep is not None
                and ep + 1 >= PROTOCOL['min_epochs']
                and ep + 1 - best_ep >= PROTOCOL['patience']):
            stop_by = 'patience'
            print('    no new best since epoch %d; stopping at %d/%d with %d mistakes'
                  % (best_ep, ep + 1, epochs, m), flush=True)
            break
    final_m = stop_mistakes(policy, stop_items) if stop_items is not None else None
    kept = 'last'
    if best_sd is not None and final_m is not None and best_m < final_m:
        policy.load_state_dict(best_sd)
        kept = 'incumbent' if best_ep == start else 'best@%d' % best_ep
        print('    kept %s: %d mistakes against the last epoch\'s %d'
              % (kept, best_m, final_m), flush=True)
        final_m = best_m
    return last, ran, {'entry': entry_m, 'best': best_m, 'final': final_m, 'kept': kept,
                       'stop': stop_by, 'trace': trace}


def adopt_uploaded():
    """A zip from an earlier session carries `state.pt`; the run continues from it."""
    target = pathlib.Path(ROOT, 'runs', ARM, 'state.pt')
    if target.exists() or not CONTENT.is_dir():
        return False
    for z in sorted(CONTENT.glob('*results*.zip')) + sorted(CONTENT.glob('*.zip')):
        try:
            with zipfile.ZipFile(z) as fh:
                names = [n for n in fh.namelist() if n.startswith('runs/%s/' % ARM)]
                if not any(n.endswith('state.pt') for n in names):
                    continue
                for n in names:
                    if '..' in pathlib.PurePosixPath(n).parts:
                        continue
                    out = pathlib.Path(ROOT, n)
                    out.parent.mkdir(parents=True, exist_ok=True)
                    out.write_bytes(fh.read(n))
        except zipfile.BadZipFile:
            continue
        print('state recovered from the uploaded zip:', z.name, '->', target)
        return True
    return False


def write_json(complete, stop_reason, base, log, agg, cert, elapsed):
    final = {k: v for k, v in cert.items() if k != 'per_root'}
    doc = {
        'args': {'rows': 6, 'cols': 7, 'win': 4, 'empties': E, 'roots': 40,
                 'roots_file': SUITE, **BUDGET, 'seed': SEED, 'threads': 1,
                 'no_controls': True, 'out': str(OUT_JSON.relative_to(ROOT)),
                 'ckpt_out': str(CKPT.parent.relative_to(ROOT))},
        'protocol': dict(PROTOCOL),
        'environment': {'python': platform.python_version(), 'torch': torch.__version__,
                        'cuda': torch.version.cuda, 'device': DEVICE,
                        'deterministic': False},
        'game': {'rows': g.ROWS, 'cols': g.COLS, 'win': g.K, 'empties': E,
                 'roots': ST['suite']['roots'],
                 'root_value_0': ST['suite']['value_0'],
                 'root_value_1': ST['suite']['value_1']},
        'replay_histories': ST['replay_histories'],
        'replay_items': ST['replay_items'],
        'suite': ST['suite'],
        'base': {'certificate': base, 'fit_loss': ST['base_loss'],
                 'fit_epochs': ST['base_epochs'], 'source': 'distilled on the replay set'},
        'repair': {'log': log,
                   'fits': ST.get('fits', []),
                   'final': final,
                   'per_root': cert.get('per_root', []),
                   'aggregate': [list(h) for h in agg],
                   'checkpoint': str(CKPT.relative_to(ROOT)) if CKPT.exists() else '',
                   'base_checkpoint': str(BASE_CKPT.relative_to(ROOT))
                   if BASE_CKPT.exists() else '',
                   'elapsed_s': round(elapsed, 1)},
        'controls': [],
        'complete': complete,
        'stop': stop_reason,
        'provenance': {'notebook': 'colab/c4_end14_t4.ipynb',
                       'bundle_commit': globals().get('BUNDLE_COMMIT', 'unknown'),
                       'sessions': ST.get('sessions', []),
                       'work_s': round(ST.get('work_s', 0.0) + elapsed, 1)},
        'elapsed_s': round(elapsed, 1),
    }
    OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
    OUT_JSON.write_text(json.dumps(doc, indent=2), encoding='utf-8')
    return doc


def run_arm(minutes=600):
    """Run, continue or resume the arm until it certifies, stalls, or the budget ends."""
    if not torch.cuda.is_available():
        raise SystemExit('this notebook exists to move the refits onto a GPU; with no GPU '
                         'the same work is better done by the laptop arms already running')
    adopt_uploaded()
    t_start = time.time()
    deadline = t_start + minutes * 60 if minutes else None
    oracle = g.Oracle()
    roots = json.loads((ROOT / SUITE).read_text(encoding='utf-8'))
    digest = hashlib.sha256((ROOT / SUITE).read_bytes()).hexdigest()
    claimed = globals().get('BUNDLE_FILES', {}).get(SUITE)
    if claimed and claimed != digest:
        raise SystemExit('the suite on disk is not the one the bundle records, so this arm '
                         'would not be certified over the published roots')
    print('suite %s: %d roots, sha256 %s%s'
          % (SUITE, len(roots), digest[:16], ' (the bundled published suite)' if claimed else ''),
          flush=True)
    order, rng = tuple(range(g.COLS)), random.Random(SEED + 1)
    hist, seen = [], set()
    for r in roots:
        for h in root_items(r, oracle, order, rng, BUDGET['branch']):
            if h not in seen:
                seen.add(h)
                hist.append(h)
    replay = items_from(hist, oracle)
    print('replay: %d histories, %d training items' % (len(hist), len(replay)), flush=True)

    have = STATE_PATH.exists()
    if have:
        ST.update(torch.load(STATE_PATH, map_location='cpu', weights_only=False))
        print('state on disk: phase %s, round %s, %.0f min of work behind it'
              % (ST.get('phase'), ST.get('next_round'), ST.get('work_s', 0) / 60), flush=True)
    ST['suite'] = {'path': SUITE, 'sha256': digest, 'roots': len(roots),
                   'value_0': [r['value'] for r in roots].count(0),
                   'value_1': [r['value'] for r in roots].count(1),
                   'in_bundle': bool(claimed)}
    ST['replay_histories'] = len(hist)
    ST['replay_items'] = len(replay)
    ST.setdefault('sessions', []).append({'started_utc': now(), 'device': DEVICE,
                                          'torch': torch.__version__,
                                          'round': ST.get('next_round', 0)})

    if not have:
        torch.manual_seed(SEED)     # the engine's init; the same draw on CPU and on the GPU
    policy = GRUPolicy(BUDGET['hidden'], BUDGET['emb']).to(DEV)

    if ST.get('base') is None:
        t0 = time.time()
        try:
            loss, eps, _ = fit_res(policy, replay, BUDGET['epochs'], BUDGET['lr'], SEED, replay,
                                   tag='base', deadline=deadline)
        except Paused as exc:
            print('PAUSED in the base fit: %s' % exc, flush=True)
            ST['sessions'][-1].update(ended_utc=now(), round=0, stop='clock',
                                      minutes=round((time.time() - t_start) / 60, 1))
            ST['work_s'] = ST.get('work_s', 0.0) + (time.time() - t_start)
            save()
            print('the base fit is on disk at its last epoch; run this cell again to continue',
                  flush=True)
            return {'complete': False, 'stop': 'clock', 'round': 0}
        twin = twin_of(policy)
        base = {k: v for k, v in certify(twin, oracle, roots).items() if k != 'per_root'}
        BASE_CKPT.parent.mkdir(parents=True, exist_ok=True)
        policy0 = cpu_state(policy)
        torch.save(policy0, str(BASE_CKPT))
        ST.update(base=base, base_loss=loss, base_epochs=eps, policy0=policy0,
                  next_round=0, log=[], agg=[], fits=[], model=cpu_state(policy),
                  phase='round')
        save()
        print('base fit loss %.4f epochs %d | W %.0f B %.0f D %.4f viol %d [%.0f min]'
              % (loss, eps, base['worst_draw_rate'], base['best_draw_rate'],
                 base['random_draw_rate_exact'], base['n_violations'],
                 (time.time() - t0) / 60), flush=True)
    else:
        policy.load_state_dict(ST['model'])
        twin = twin_of(policy)
        base = ST['base']
        print('base already fitted: W %.0f B %.0f D %.4f viol %d'
              % (base['worst_draw_rate'], base['best_draw_rate'],
                 base['random_draw_rate_exact'], base['n_violations']), flush=True)

    it, log, agg = ST['next_round'], ST['log'], [tuple(h) for h in ST['agg']]
    cert = certify(twin, oracle, roots)
    logged = {e['round'] for e in log}
    stop_reason = 'max_iters'
    while it < BUDGET['max_iters']:
        if it not in logged:
            replay_mis = n_optimal_mistakes(twin, replay)
            log.append({'round': it,
                        'worst_draw_rate': cert['worst_draw_rate'],
                        'best_draw_rate': cert['best_draw_rate'],
                        'random_draw_rate_exact': cert['random_draw_rate_exact'],
                        'n_violations': cert['n_violations'],
                        'n_roots_drawn_worst': cert['n_roots_drawn_worst'],
                        'aggregate': len(agg),
                        'replay_mistakes': replay_mis})
            print('[repair it %2d] W %.0f B %.0f D %.4f drawn %d/%d viol %3d agg %3d '
                  'rep_mis %d' % (it, cert['worst_draw_rate'], cert['best_draw_rate'],
                                  cert['random_draw_rate_exact'],
                                  cert['n_roots_drawn_worst'], cert['n_roots'],
                                  cert['n_violations'], len(agg), replay_mis), flush=True)
            logged.add(it)
            ST.update(log=log, agg=[list(h) for h in agg], next_round=it, phase='round')
            write_json(False, 'running', base, log, agg, cert, time.time() - t_start)
            save()
        if cert['worst_draw_rate'] == 1.0:
            stop_reason = 'certified'
            break
        if not (ST.get('phase') == 'fit' and ST.get('tag') == it + 1):
            known = set(agg)
            new = [tuple(v['history']) for v in cert['violations']
                   if tuple(v['history']) not in known]
            if not new:
                stop_reason = 'no new violations'
                break
            agg.extend(new)
        agg_items = items_from(agg, oracle)
        stop = (replay + agg_items if BUDGET['stop'] == 'both'
                else agg_items if BUDGET['stop'] == 'agg' else replay)
        train = replay + agg_items * BUDGET['agg_repeat']
        t1 = time.time()
        try:
            loss, eps, info = fit_res(policy, train, BUDGET['epochs'], BUDGET['lr'],
                                      SEED + it + 1, stop, tag=it + 1, deadline=deadline)
        except Paused as exc:
            print('PAUSED: %s' % exc, flush=True)
            stop_reason = 'clock'
            break
        twin = twin_of(policy)
        cert = certify(twin, oracle, roots)
        ST.setdefault('fits', []).append({'round': it + 1, 'epochs': eps,
                                          'loss': round(loss, 6),
                                          'items': len(train),
                                          'batches': -(-len(train) // 256),
                                          'agg_items': len(agg_items),
                                          'agg_mistakes': n_optimal_mistakes(twin, agg_items),
                                          'entry_mistakes': info['entry'],
                                          'best_mistakes': info['best'],
                                          'kept': info['kept'],
                                          'stopped_by': info['stop'],
                                          'mistake_trace': info['trace'],
                                          'seconds': round(time.time() - t1, 1)})
        it += 1
        ST.update(log=log, agg=[list(h) for h in agg], next_round=it,
                  model=cpu_state(policy), phase='round')
        write_json(False, 'running', base, log, agg, cert, time.time() - t_start)
        save()
        # `ST['model']` is the policy the certificate above was computed on, so it is also the
        # only weights that belong beside that certificate. Writing them here keeps an arm that
        # stops on the clock readable at its last completed round: the final block below must
        # never fall back to `ST['model']`, which by then is mid-fit and uncertified.
        CKPT.parent.mkdir(parents=True, exist_ok=True)
        torch.save(ST['model'], str(CKPT))
        print('    fit loss %.4f epochs %d agg_mistakes %d kept %s stopped %s [%.0f min] | '
              '%d items, %d batches'
              % (loss, eps, ST['fits'][-1]['agg_mistakes'], info['kept'], info['stop'],
                 (time.time() - t1) / 60, len(train), -(-len(train) // 256)), flush=True)
        if deadline and time.time() > deadline:
            stop_reason = 'clock'
            break

    if stop_reason in ('certified', 'no new violations', 'max_iters'):
        CKPT.parent.mkdir(parents=True, exist_ok=True)
        torch.save(ST['model'], str(CKPT))
    complete = stop_reason in ('certified', 'no new violations', 'max_iters')
    ST['sessions'][-1].update(ended_utc=now(), round=it, stop=stop_reason,
                              minutes=round((time.time() - t_start) / 60, 1))
    # write_json adds this session's time to ST['work_s'] itself; adding it here as well puts
    # every session into the JSON twice, so the total is stored after the JSON is written.
    doc = write_json(complete, stop_reason, base, log, agg, cert, time.time() - t_start)
    ST['work_s'] = ST.get('work_s', 0.0) + (time.time() - t_start)
    save()
    print('round %d: W %.0f B %.0f D %.4f, %d aggregate histories, stop=%s, %s'
          % (it, cert['worst_draw_rate'], cert['best_draw_rate'],
             cert['random_draw_rate_exact'], len(agg), stop_reason,
             'complete' if complete else 'not complete - run this cell again to continue'),
          flush=True)
    print('wrote %s and %s' % (OUT_JSON.relative_to(ROOT), CKPT.relative_to(ROOT)), flush=True)
    return doc


print('driver ready on %s: refits fit on the GPU, the certificate runs on a CPU twin, and '
      'each refit keeps %s%s' % (DEVICE, 'its best epoch' if PROTOCOL['keep_best'] else
                                 'its last epoch (the published rule)',
                                 '' if not PROTOCOL['keep_best'] else
                                 ', stopping after %d epochs without a new best (never before '
                                 '%d)' % (PROTOCOL['patience'], PROTOCOL['min_epochs'])))


## 3. Run it

The cell below runs the two arms one after another, each for the minutes it is given and then
hands the time back to the next. Nothing here needs the tab to stay awake in a particular way: a
cell that is running is activity, so the runtime does not idle out while a fit is in progress,
and when the deadline inside `fit_res` is crossed the loop stops after the epoch in flight,
saves, and returns.

Read the round lines the way the laptop logs read, because they are the same lines:

```
[repair it 13] W 0 B 1 D 0.9631 drawn 22/40 viol  61 agg 2393 rep_mis 0
```

`W` is the worst draw rate over the forty roots against every optimal teacher, `B` the best,
`D` the uniform average; `drawn` counts the roots forced to a draw against every tie-break;
`viol` are the violating histories that will be added; `agg` is the aggregate set's size;
`rep_mis` is the number of replay items the policy now answers suboptimally. `W=1` means
certified and the loop stops there.

Two things to expect, and neither is a bug in this notebook. First, the aggregate grows by
hundreds of histories a round and every refit trains on all of it: the first fit after the
base is minutes, the fifteenth is hours, and the fit's own stopping test may be reached long
before epoch 200 or not at all. Second, none of these arms can fall back the way the laptop
seeds did: those logs show seed 1 at `D 0.9742` on round 17 and `0.1886` on round 19 because
its refit kept its last epoch, and here a refit that ends with more mistakes than it started
with hands the arm back its own weights and says so in the `kept` column of the JSON. What that
buys is not a certificate, it is the absence of that fall: if the aggregate at this depth is
beyond what the fit can satisfy, an arm holds near its best `D` until `max_iters`, and that is
the answer to report.


In [ ]:
# One pass over the seed list. Each arm gets its own state, JSON, weights and zip; the
# constants below are rebound per arm and ST is the state `save`/`run_arm` read. An uploaded
# zip of one of these two arms carries runs/<arm>/state.pt, so its fit continues instead of
# restarting. Seeds 3 and 1 are not in the list: both are finished (round 32 at the wall, round
# 24 at the cap) and resuming them would certify the same weights and stop again.
RUNS = [('c4_end14_t4_s0', 0, 24, 75),
        ('c4_end14_t4_s2', 2, 24, 75)]
for _arm, _seed, _iters, _minutes in RUNS:
    globals().update(ARM=_arm, SEED=_seed, ARM_STEM=_arm,
                     SLUG='t4_%s' % _arm.rsplit('_', 1)[-1],
                     STATE_PATH=ROOT / 'runs' / _arm / 'state.pt',
                     OUT_JSON=ROOT / 'data' / (_arm + '.json'),
                     CKPT=ROOT / 'runs' / _arm / 'c4_gru.pt',
                     BASE_CKPT=ROOT / 'runs' / _arm / 'c4_gru_base.pt')
    ST.clear()
    BUDGET['max_iters'] = _iters
    adopt_uploaded()
    print('\n=== %s: seed %d, %d rounds, %d min ===' % (_arm, _seed, _iters, _minutes),
          flush=True)
    try:
        run_arm(minutes=_minutes)
    finally:
        ship()


## 4. The two paired reads, once an arm finishes

The queue does not stop at the repair arm: every arm in the paper is followed by an independent
verification on suite A and a read on the blind suite B, both from the saved weights, by
`src/c4_endgame_verify.py` with `--orders 120 --games 40`. That is what makes a fourteen-empty
row comparable with the depth-12 rows, and it is cheap next to the arm. The cell goes over the
two arms in order, refuses each one that is not complete, because a read taken from a
half-finished repair is a read of a policy that will not exist, and ships that arm's zip again
once its reads are in.


In [ ]:
import json, subprocess, sys

for _arm in ('c4_end14_t4_s0', 'c4_end14_t4_s2'):
    _json = ROOT / 'data' / (_arm + '.json')
    _ckpt = ROOT / 'runs' / _arm / 'c4_gru.pt'
    if not _json.exists() or not _ckpt.exists():
        print('%s: no JSON or no weights yet; nothing to read' % _arm, flush=True)
        continue
    d = json.loads(_json.read_text(encoding='utf-8'))
    last = d['repair']['log'][-1]
    print('\n%s: complete=%s stop=%s round=%d W=%.0f'
          % (_arm, d['complete'], d['stop'], last['round'], last['worst_draw_rate']),
          flush=True)
    if not d['complete']:
        print('  not complete: its reads wait for the repair to end', flush=True)
        continue
    for _suite, _name in ((SUITE, _arm + '_verify.json'),
                          (SUITE_B, _arm + '_holdoutB.json')):
        _cmd = [sys.executable, 'src/c4_endgame_verify.py',
                '--ckpt', str(_ckpt), '--roots', _suite,
                '--hidden', str(BUDGET['hidden']), '--emb', str(BUDGET['emb']),
                '--orders', '120', '--games', '40',
                '--out', str(ROOT / 'data' / _name)]
        print('$ ' + ' '.join(_cmd), flush=True)
        subprocess.run(_cmd, cwd=str(ROOT), check=True)
        v = json.loads((ROOT / 'data' / _name).read_text(encoding='utf-8'))
        print('  %s: W %.0f B %.0f D %.4f roots %d'
              % (_name, v['worst_draw_rate'], v['best_draw_rate'],
                 v['random_draw_rate_exact'], v['roots']), flush=True)
    globals().update(ARM=_arm, ARM_STEM=_arm,
                     SLUG='t4_%s' % _arm.rsplit('_', 1)[-1],
                     OUT_JSON=ROOT / 'data' / (_arm + '.json'), CKPT=_ckpt)
    ship()
print('reads done; each arm that finished has its zip in the browser')


## 5. Getting the work out of the runtime, and getting it back in

`ship()` zips what exists for the current arm and hands it to the browser, one archive per arm
(`c4_t4_s0_results.zip` and `c4_t4_s2_results.zip`). That zip is the only exit; nothing is
mounted and nothing is uploaded, so if the tab is closed before `ship()` runs, `state.pt` goes
with the runtime. The cell that runs the arms already calls it after each one, and calling it
again is cheap when nothing changed.

To continue in a new session: upload the zips through the file sidebar (they land in
`/content`), then Runtime -> Run all. The driver finds each `runs/<arm>/state.pt` inside its
zip, restores it, and picks up at the same round and the same epoch, with the optimizer's
moments and the shuffle order - the interrupted fit continues rather than restarting, which is
the difference between losing five epochs and losing fifty hours. The best epoch that refit had
already scored travels in the same file, so a session boundary cannot cost the rule the
incumbent it was holding. An arm whose zip is not uploaded starts from its base fit. Seeds 3 and 1
are not in the list at all: both are finished, their zips would only re-certify the weights that
stopped (round 32 at the wall, round 24 at the cap), and a notebook pass cannot add a round the
loop has no history to add.

What a session looks like when it ends without an arm finishing: each JSON is written at every
round with `"complete": false`, so the newest file in each zip always states how far that arm
got, on which device, in which sessions, and how much wall time it has consumed. That is the
record the paper needs if a run is ever cited as an unfinished arm rather than a certificate.
